In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
# -----------------------------
# 1. Load and prepare data
# -----------------------------
iris = load_iris()
X = iris.data          # (150, 4)
y = iris.target        # (150,)
# Binary classification: Setosa vs Non-Setosa
y = (y == 0).astype(np.float32)  # Setosa = 1, others = 0

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print("X_train (first 5 samples):")
print(X_train[:5])

print("\ny_train (first 5 labels):")
print(y_train[:5])

print("\nX_test (first 5 samples):")
print(X_test[:5])

print("\ny_test (first 5 labels):")
print(y_test[:5])

X_train (first 5 samples):
[[4.6 3.6 1.  0.2]
 [5.7 4.4 1.5 0.4]
 [6.7 3.1 4.4 1.4]
 [4.8 3.4 1.6 0.2]
 [4.4 3.2 1.3 0.2]]

y_train (first 5 labels):
[1. 1. 0. 1. 1.]

X_test (first 5 samples):
[[6.1 2.8 4.7 1.2]
 [5.7 3.8 1.7 0.3]
 [7.7 2.6 6.9 2.3]
 [6.  2.9 4.5 1.5]
 [6.8 2.8 4.8 1.4]]

y_test (first 5 labels):
[0. 1. 0. 0. 0.]


In [ ]:
# Feature scaling (important for gradient descent)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [ ]:
# -----------------------------
# 2. Define a single neuron model
# -----------------------------
class SingleNeuron(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.linear(x))

model = SingleNeuron(input_dim=4)

In [ ]:
# -----------------------------
# 3. Loss and optimizer
# -----------------------------
criterion = nn.BCELoss()   # Binary Cross Entropy
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
# -----------------------------
# 4. Training loop
# -----------------------------
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


Epoch [20/100], Loss: 0.0724
Epoch [40/100], Loss: 0.0591
Epoch [60/100], Loss: 0.0500
Epoch [80/100], Loss: 0.0435
Epoch [100/100], Loss: 0.0385


In [ ]:
# -----------------------------
# 5. Evaluation
# -----------------------------
with torch.no_grad():
    probs = model(X_test)
    predictions = (probs >= 0.5).float()
    accuracy = (predictions == y_test).float().mean()

print(f"\nTest Accuracy: {accuracy.item() * 100:.2f}%")


Test Accuracy: 100.00%
